In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from datetime import datetime
import json

def get_fighter_links(): # gets the links to each fighters profile
    base_url = 'http://ufcstats.com/statistics/fighters?'
    all_links = set()
    
    for char in 'abcdefghijklmnopqrstuvwxyz':
        url = f"{base_url}char={char}&page=all"
        try:
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
            soup = BeautifulSoup(response.content, 'lxml')
            
            # Target the specific links in table rows
            for row in soup.select('tr.b-statistics__table-row'):
                link_tag = row.find('a', class_='b-link b-link_style_black')
                if link_tag and link_tag.has_attr('href'):
                    all_links.add(link_tag['href'])
                    
        except Exception as e:
            print(f"Error processing {char.upper()}: {str(e)}")
    
    return list(all_links)

# Usage
fighter_links = get_fighter_links()
print(f"Found {len(fighter_links)} unique fighter profiles")


Found 4451 unique fighter profiles


In [5]:
import requests
from bs4 import BeautifulSoup

def parse_fighter_page(url):
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(response.content, 'lxml')
        
        data = {
            'name': extract_name(soup),
            **extract_physical_stats(soup),
        }
        return data
        
    except Exception as e:
        print(f"Error parsing {url}: {str(e)}")
        return None

def extract_name(soup):
    name_tag = soup.find('span', class_='b-content__title-highlight')
    return name_tag.get_text(strip=True) if name_tag else 'Missing'

def extract_physical_stats(soup):
    # Initialize with defaults in case data is missing on the page
    stats = {
        'height': 'Missing',
        'reach': 'Missing',
        'dob': 'Missing'
    }
    
    # Target the list items that contain the stats
    items = soup.find_all('li', class_='b-list__box-list-item_type_block')
    
    for item in items:
        # The label (e.g., "Height:") is inside an <i> tag
        if item.find('i'):
            title = item.find('i').get_text(strip=True)
            # The value is the text of the <li> minus the text of the <i> label
            # We strip to remove extra whitespace/newlines
            value = item.get_text(strip=True).replace(title, '').strip()

            if 'Height:' in title:
                stats['height'] = value
            elif 'Reach:' in title:
                stats['reach'] = value
            elif 'DOB:' in title:
                stats['dob'] = value
            
    return stats

# Usage example
fighter_url = "http://ufcstats.com/fighter-details/f923e012414c883e"
print(parse_fighter_page(fighter_url))

{'name': 'Lauren Mueller', 'height': '5\' 5"', 'reach': '67"', 'dob': 'Nov 15, 1991'}


In [6]:
def scrape_all_fighters_csv(filename="ufc_fighters.csv"):
    print("Fetching fighter links...")
    links = get_fighter_links()
    all_data = []
    
    print(f"Found {len(links)} fighters. Starting scrape...")
    
    for link in links:
        try:
            # Reusing your simplified parse function from before
            data = parse_fighter_page(link)
            if data:
                data['url'] = link
                all_data.append(data)
                print(f".", end="", flush=True) # Simple progress bar
        except Exception as e:
            print(f"\nFailed: {link} | {e}")

    # Convert to DataFrame and save immediately to current directory
    if all_data:
        df = pd.DataFrame(all_data)
        df.to_csv(filename, index=False)
        print(f"\n\nSuccess! Saved {len(all_data)} fighters to '{filename}'")
    else:
        print("\nNo data collected.")

# Run it
scrape_all_fighters_csv()

Fetching fighter links...
Found 4451 fighters. Starting scrape...
......................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................